# OpenWorm Movement DB — Strain filtering + download (Zenodo)

This notebook:
1. Loads **two CSVs** you already generated:
   - `openworm_movement_zenodo_metadata.csv`
   - `openworm_parsed_all_fields.csv`
2. Lists **all strains** and their **counts** (most → least).
3. Lets you **filter** records to a single strain.
4. Uses the **Zenodo API** to **download all files** for the filtered records.

> Tip: downloads can be large. Start with a small strain first.


In [2]:
import os
import re
import time
from pathlib import Path
from typing import Dict, Any, List

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option('display.max_colwidth', 200)


## 1) Load your two CSV files

Update the paths below if your files are elsewhere.


In [3]:
import pandas as pd
import csv

METADATA_CSV = "openworm_movement_zenodo_metadata.csv"
PARSED_CSV   = "openworm_parsed_all_fields.csv"

meta = pd.read_csv(
    METADATA_CSV,
    engine="python",            # more robust than C engine
    sep=",",
    quotechar='"',
    doublequote=True,
    escapechar="\\",
    encoding="utf-8",
)

parsed = pd.read_csv(
    PARSED_CSV,
    engine="python",
    sep=",",
    quotechar='"',
    doublequote=True,
    escapechar="\\",
    encoding="utf-8",
)

print("meta rows:", len(meta), "columns:", len(meta.columns))
print("parsed rows:", len(parsed), "columns:", len(parsed.columns))
parsed.head(2)

meta rows: 15001 columns: 16
parsed rows: 15001 columns: 30


,allele,arena,base_name,chromosome,days_of_adulthood,featureid,food,frames_per_second,gene,habituation,...,stage,strain,strain_description,timestamp,total_time_s,ventral_side,version,video_micrometers_per_pixel,who,worm_id
0,NaN,style : petri,flp-4 (yu35) on food R_2013_02_28__13_08_32___5___7,II,NaN,@OMG,OP50,30.0300,NaN,30m wait,...,adult,NY119,flp-4(yn35),2013-02-28T13:08:32+00:00,898.967,anticlockwise,cbfc23eb4f1ac2f29be75ade7a937eed58a5b219,4.57165,Laura Grundy,NaN
1,hs13,style : petri,nc-127 (hs13)V on food L_2010_04_14__10_20_12___7___2,V,NaN,@OMG,OP50,31.9714,unc-127,30m wait,...,adult,HH27,unc-127(hs13)V,2010-04-14T10:20:12+01:00,899.962,clockwise,cbfc23eb4f1ac2f29be75ade7a937eed58a5b219,4.31067,Laura Grundy,NaN


## 2) Merge the two tables

We need a record identifier to download from Zenodo.

- If `parsed` already has `record_id`, we merge on it.
- Otherwise, we assume **row order matches** the metadata CSV (because `parsed` was built from `df.description`).
  In that case we copy `meta['record_id']` into `parsed`.


In [6]:
if 'record_id' in parsed.columns:
    merged = parsed.merge(meta[['record_id','doi','title','publication_date']], on='record_id', how='left')
else:
    if len(parsed) != len(meta):
        raise ValueError("parsed and meta row counts differ; cannot align by row order. Add record_id to parsed when exporting.")
    parsed = parsed.copy()
    parsed['record_id'] = meta['record_id'].values
    merged = parsed.merge(meta[['record_id','doi','title','publication_date']], on='record_id', how='left')

# Normalize strain
merged["strain_norm"] = merged["strain"].astype(str).str.strip().str.lower()
merged.loc[
    merged["strain_norm"].isin(["nan", "-n/a-", "n/a", "na", "-", ""]),
    "strain_norm"
] = pd.NA

print("merged rows:", len(merged))
merged[['record_id','title','strain','stage','sex','total_time_s','food', 'frames_per_second']].head(5)

merged rows: 15001


,record_id,title,strain,stage,sex,total_time_s,food,frames_per_second
0,1006899,NY119 flp-4(yn35) | 2013-02-28T13:08:32+00:00,NY119,adult,hermaphrodite,898.967,OP50,30.0300
1,1011609,HH27 unc-127(hs13)V | 2010-04-14T10:20:12+01:00,HH27,adult,hermaphrodite,899.962,OP50,31.9714
2,1021991,CB4371 unc-118(e2331)X | 2010-04-14T10:41:03+01:00,CB4371,adult,hermaphrodite,899.218,OP50,31.9489
3,1011762,CB193 unc-29(e193)I | 2009-09-16T14:56:02+01:00,CB193,adult,hermaphrodite,1797.290,OP50,25.7732
4,1023519,RB1250 acr-21(ok1314)III | 2009-07-24T12:27:17+01:00,RB1250,adult,hermaphrodite,299.647,OP50,25.7069


## 3) Strain counts (most → least)


In [70]:
import pandas as pd

# ---- Set your time thresholds ----
MIN_TIME = 700
MAX_TIME = 900

# Ensure required columns exist
if 'strain' not in merged.columns:
    raise ValueError("No 'strain' column found in merged data.")

if 'total_time_s' not in merged.columns:
    raise ValueError("No 'total_time_s' column found in merged data.")

# Ensure total_time_s is numeric
merged["total_time_s"] = pd.to_numeric(merged["total_time_s"], errors="coerce")

# Normalize strain
merged["strain_norm"] = merged["strain"].astype(str).str.strip().str.lower()
merged.loc[
    merged["strain_norm"].isin(["nan", "-n/a-", "n/a", "na", "-", ""]),
    "strain_norm"
] = pd.NA

# ---- Filter by time range ----
filtered_time = merged[
    merged["total_time_s"].between(MIN_TIME, MAX_TIME) &
    (merged["strain_norm"].notna())
].copy()

# Compute counts
strain_counts = filtered_time["strain_norm"].value_counts()
strain_counts_df = strain_counts.rename_axis("strain").reset_index(name="count")

print(f"Time range: {MIN_TIME} to {MAX_TIME} seconds")
print("Unique strains (after filter):", len(strain_counts_df))

display(strain_counts_df.head(50))

# Save
output_file = f"strain_count_time_{MIN_TIME}_to_{MAX_TIME}.csv"
strain_counts_df.to_csv(output_file, index=False)

output_file

Time range: 700 to 900 seconds
Unique strains (after filter): 384


,strain,count
0,n2,1535
1,ow953,463
2,aq2947,456
3,ow949,405
4,ow940,399
5,ow956,390
6,ow939,357
7,cb1112,317
8,cb5,201
9,rb557,75


'strain_count_time_700_to_900.csv'

In [71]:
# For each strain: counts of stage and sex (and a combined stage×sex table)

import pandas as pd

# Ensure normalized strain exists
if "strain_norm" not in merged.columns:
    merged["strain_norm"] = merged["strain"].astype(str).str.strip().str.lower()
    merged.loc[merged["strain_norm"].isin(["nan","-n/a-","n/a","na","-",""]), "strain_norm"] = pd.NA

# Optional: normalize stage/sex similarly
merged["stage_norm"] = merged["stage"].astype(str).str.strip().str.lower()
merged.loc[merged["stage_norm"].isin(["nan","-n/a-","n/a","na","-",""]), "stage_norm"] = pd.NA

merged["sex_norm"] = merged["sex"].astype(str).str.strip().str.lower()
merged.loc[merged["sex_norm"].isin(["nan","-n/a-","n/a","na","-",""]), "sex_norm"] = pd.NA


# 1) For each strain: stage counts (wide table)
stage_by_strain = (
    merged.dropna(subset=["strain_norm", "stage_norm"])
          .groupby(["strain_norm", "stage_norm"])
          .size()
          .unstack(fill_value=0)
          .sort_index()
)

# 2) For each strain: sex counts (wide table)
sex_by_strain = (
    merged.dropna(subset=["strain_norm", "sex_norm"])
          .groupby(["strain_norm", "sex_norm"])
          .size()
          .unstack(fill_value=0)
          .sort_index()
)

# 3) For each strain: stage × sex counts (long table)
stage_sex_by_strain = (
    merged.dropna(subset=["strain_norm", "stage_norm", "sex_norm"])
          .groupby(["strain_norm", "stage_norm", "sex_norm"])
          .size()
          .reset_index(name="count")
          .sort_values(["strain_norm", "count"], ascending=[True, False])
)

display(stage_by_strain.head(20))
display(sex_by_strain.head(20))
display(stage_sex_by_strain.head(50))

# Optional saves
stage_by_strain.to_csv("stage_counts_by_strain.csv")
sex_by_strain.to_csv("sex_counts_by_strain.csv")
stage_sex_by_strain.to_csv("stage_sex_counts_by_strain.csv", index=False)

stage_norm,adult,l3,l4
strain_norm,,,
aq1031,32,0,0
aq1033,50,0,0
aq1037,59,0,0
aq1038,51,0,0
aq1413,46,0,0
aq1422,28,0,0
aq1441,25,0,0
aq2056,21,0,0
aq2153,29,0,0


sex_norm,hermaphrodite,male
strain_norm,,
aq1031,32,0
aq1033,50,0
aq1037,59,0
aq1038,51,0
aq1413,46,0
aq1422,28,0
aq1441,25,0
aq2056,21,0
aq2153,29,0


,strain_norm,stage_norm,sex_norm,count
0,aq1031,adult,hermaphrodite,32
1,aq1033,adult,hermaphrodite,50
2,aq1037,adult,hermaphrodite,59
3,aq1038,adult,hermaphrodite,51
4,aq1413,adult,hermaphrodite,46
5,aq1422,adult,hermaphrodite,28
6,aq1441,adult,hermaphrodite,25
7,aq2056,adult,hermaphrodite,21
8,aq2153,adult,hermaphrodite,29
9,aq2197,adult,hermaphrodite,23


In [72]:
import pandas as pd
import numpy as np

# Ensure total_time_s is numeric
merged["total_time_s"] = pd.to_numeric(merged["total_time_s"], errors="coerce")

# Drop rows without strain or time
time_df = merged.dropna(subset=["strain_norm", "total_time_s"]).copy()

# Define bin size
BIN_SIZE = 100

# Compute bins dynamically based on max time
max_time = time_df["total_time_s"].max()
bins = np.arange(0, max_time + BIN_SIZE, BIN_SIZE)

# Create bucket labels like "0-99", "100-199", etc.
labels = [f"{int(b)}-{int(b+BIN_SIZE-1)}" for b in bins[:-1]]

time_df["time_bucket"] = pd.cut(
    time_df["total_time_s"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

# 1) Wide table: strain × time bucket
strain_time_table = (
    time_df.groupby(["strain_norm", "time_bucket"])
           .size()
           .unstack(fill_value=0)
           .sort_index()
)

display(strain_time_table.head(20))

# 2) Long format (useful for plotting / ML)
strain_time_long = (
    time_df.groupby(["strain_norm", "time_bucket"])
           .size()
           .reset_index(name="count")
           .sort_values(["strain_norm", "time_bucket"])
)

display(strain_time_long.head(50))

# Save outputs
strain_time_table.to_csv("strain_time_bucket_counts_wide.csv")
strain_time_long.to_csv("strain_time_bucket_counts_long.csv", index=False)

time_bucket,0-99,100-199,200-299,300-399,400-499,500-599,600-699,700-799,800-899,900-999,...,4400-4499,6500-6599,6800-6899,7100-7199,10700-10799,13900-13999,16900-16999,21500-21599,28700-28799,35900-35999
strain_norm,,,,,,,,,,,,,,,,,,,,,
aq1031,0,0,0,0,0,0,0,0,32,0,...,0,0,0,0,0,0,0,0,0,0
aq1033,0,0,0,0,0,0,0,0,50,0,...,0,0,0,0,0,0,0,0,0,0
aq1037,0,0,0,0,0,0,0,0,59,0,...,0,0,0,0,0,0,0,0,0,0
aq1038,1,0,0,0,0,0,0,0,50,0,...,0,0,0,0,0,0,0,0,0,0
aq1413,0,0,0,0,0,0,0,0,46,0,...,0,0,0,0,0,0,0,0,0,0
aq1422,4,5,0,0,0,0,0,0,19,0,...,0,0,0,0,0,0,0,0,0,0
aq1441,0,1,0,0,0,0,0,0,24,0,...,0,0,0,0,0,0,0,0,0,0
aq2056,0,0,0,0,0,0,0,0,21,0,...,0,0,0,0,0,0,0,0,0,0
aq2153,0,0,0,0,0,0,0,0,28,0,...,0,0,0,0,0,0,0,0,0,0


,strain_norm,time_bucket,count
0,aq1031,800-899,32
1,aq1033,800-899,50
2,aq1037,800-899,59
3,aq1038,0-99,1
4,aq1038,800-899,50
5,aq1413,800-899,46
6,aq1422,0-99,4
7,aq1422,100-199,5
8,aq1422,800-899,19
9,aq1441,100-199,1


In [12]:
import pandas as pd
import numpy as np

# Ensure total_time_s is numeric
merged["frames_per_second"] = pd.to_numeric(merged["frames_per_second"], errors="coerce")

# Drop rows without strain or time
time_df = merged.dropna(subset=["strain_norm", "frames_per_second"]).copy()

# Define bin size
BIN_SIZE = 4

# Compute bins dynamically based on max time
max_time = time_df["frames_per_second"].max()
bins = np.arange(0, max_time + BIN_SIZE, BIN_SIZE)

# Create bucket labels like "0-99", "100-199", etc.
labels = [f"{int(b)}-{int(b+BIN_SIZE-1)}" for b in bins[:-1]]

time_df["time_bucket"] = pd.cut(
    time_df["frames_per_second"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

# 1) Wide table: strain × time bucket
strain_time_table = (
    time_df.groupby(["strain_norm", "time_bucket"])
           .size()
           .unstack(fill_value=0)
           .sort_index()
)

##display(strain_time_table.head(20))

# 2) Long format (useful for plotting / ML)
strain_time_long = (
    time_df.groupby(["strain_norm", "time_bucket"])
           .size()
           .reset_index(name="count")
           .sort_values(["strain_norm", "time_bucket"])
)

#display(strain_time_long.head(50))

strain_time_table_sorted = (
    strain_time_table
        .sort_values(by="28-31", ascending=False)
)

display(strain_time_table_sorted.head(20))

# Save outputs
#strain_time_table.to_csv("strain_fps_bucket_counts_wide.csv")
#strain_time_long.to_csv("strain_fps_bucket_counts_long.csv", index=False)

time_bucket,4-7,12-15,16-19,20-23,24-27,28-31,32-35,36-39
strain_norm,,,,,,,,
n2,6,2,18,96,625,1026,19,0
cb5,0,0,0,7,37,160,0,0
aq2947,0,0,1,2,372,83,0,0
ps312,0,2,0,12,9,82,0,0
ow940,0,0,0,0,337,62,0,0
ed3054,0,0,0,0,1,62,0,0
vc12,0,0,0,2,4,56,0,0
aq1037,0,0,0,0,4,55,0,0
ju438,0,2,0,0,0,54,0,0


## 4) Filter to one strain

Set `TARGET_STRAIN` to the strain you want (use the lowercase form shown in the table above).


In [82]:
TARGET_STRAIN = "cb5"
TARGET_SEX = "hermaphrodite"
TARGET_STAGE = "adult"
TARGET_FOOD = "OP50"
MIN_TIME = 700
MAX_TIME = 1000

filtered = merged[
        (merged['strain_norm'] == TARGET_STRAIN) &
        (merged['stage'] == TARGET_STAGE) &
        (merged['food'] == TARGET_FOOD) &
        (merged['total_time_s'].between(MIN_TIME, MAX_TIME)) &
        (merged['sex'] == TARGET_SEX) 
    ].copy()

## 5) Download all files for the filtered records (Zenodo API)

This downloads **every file attached** to each Zenodo record.

- Files will be saved under: `downloads/<strain>/record_<record_id>/...`
- If a file already exists, it will be skipped.


In [50]:
ZENODO_RECORD_API = "https://zenodo.org/api/records/{}"

def zenodo_get_json(url: str, params: Dict[str, Any] | None = None, *, retries: int = 6, backoff: float = 1.7) -> Dict[str, Any]:
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params or {}, timeout=60)
            if r.status_code == 429:
                time.sleep(backoff ** (attempt + 1))
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(backoff ** (attempt + 1))
    raise RuntimeError(f"Zenodo GET failed after {retries} retries: {last}")


def download_file(url: str, out_path: Path, *, overwrite: bool = False):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not overwrite:
        return
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


def download_record(record_id: int, base_dir: Path, *, overwrite: bool = False) -> List[Path]:
    rec = zenodo_get_json(ZENODO_RECORD_API.format(record_id))
    files = rec.get('files', []) or []
    saved: List[Path] = []
    for f in files:
        key = f.get('key')
        url = (f.get('links') or {}).get('self')
        if not key or not url:
            continue
        out_path = base_dir / f"record_{record_id}" / key
        download_file(url, out_path, overwrite=overwrite)
        saved.append(out_path)
    return saved


In [83]:
from pathlib import Path
from tqdm.auto import tqdm

# --- Windows output root (your requested folder) ---
BASE_OUT = Path(r"C:\Github\CSCI-198-Nematode-Classification\openworm")
TARGET_DIR = BASE_OUT / TARGET_STRAIN
TARGET_DIR.mkdir(parents=True, exist_ok=True)

# --- batching controls ---
MAX_RECORDS_TO_DOWNLOAD = 10   # <-- set cap per run (batch size). Use None for no cap.
START_INDEX = 0               # <-- set to 0 for first batch; set to 50 for next batch, etc.

# Record IDs to download (sorted so batching is stable/reproducible)
record_ids = (
    filtered["record_id"]
    .dropna()
    .astype(int)
    .sort_values()
    .tolist()
)

# Apply batching
end_index = len(record_ids) if MAX_RECORDS_TO_DOWNLOAD is None else min(len(record_ids), START_INDEX + MAX_RECORDS_TO_DOWNLOAD)
batch_ids = record_ids[START_INDEX:end_index]

print(f"Total matching records for strain={TARGET_STRAIN}: {len(record_ids)}")
print(f"Downloading batch records [{START_INDEX}:{end_index}] (count={len(batch_ids)})")
print(f"Saving under: {TARGET_DIR.resolve()}")

all_saved = []
for rid in tqdm(batch_ids, desc=f"Downloading Zenodo records for strain={TARGET_STRAIN}", unit="record"):
    try:
        # Each record will be saved under:
        # C:\Github\CSCI-198-Nematode-Classification\openworm\<strain>\record_<id>\...
        saved = download_record(rid, TARGET_DIR, overwrite=False)
        all_saved.extend(saved)
    except Exception as e:
        print(f"\n[WARN] record {rid} failed: {e}")

print("\nDownloaded files:", len(all_saved))
print("Saved under:", TARGET_DIR.resolve())
print(f"Next batch START_INDEX suggestion: {end_index}")

Total matching records for strain=cb5: 201
Saving under: C:\Github\CSCI-198-Nematode-Classification\openworm\cb5



Downloaded files: 30
Saved under: C:\Github\CSCI-198-Nematode-Classification\openworm\cb5
Next batch START_INDEX suggestion: 10


## 6) Save your filtered table (optional)


In [76]:
filtered.to_csv(f"filtered_{TARGET_STRAIN}.csv", index=False)
f"filtered_{TARGET_STRAIN}.csv"

'filtered_N2.csv'